In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJ = '/content/drive/MyDrive/severity_queue'
import pandas as pd, numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(f'{PROJ}/nhamcs_2022_camp_relabeled_v2.csv')
print(f"Records: {len(df):,}")
print(f"\nLabel distribution:")
print(df['camp_triage_label_final'].value_counts().to_string())
print(f"\nColumns: {list(df.columns)}")

Mounted at /content/drive
Records: 7,578

Label distribution:
camp_triage_label_final
LOW       3124
MEDIUM    2472
HIGH      1982

Columns: ['model_input_text', 'symptoms', 'age', 'severity', 'three_level_target', 'source_severity', 'condition_group', 'vital_hr', 'vital_spo2', 'vital_sbp', 'vital_rr', 'vital_temp', 'pain_score', 'has_vitals', 'injury_flag', 'data_source', 'immedr_raw', 'dx_cpr', 'dx_intubate', 'dx_bpap', 'dx_centline', 'dx_lactate', 'dx_cardenz', 'dx_abg', 'dx_cardmon', 'dx_ekg', 'dx_ivfluids', 'dx_cbc', 'dx_catscan', 'dx_resus', 'rf_chest_pain', 'rf_breathing_difficulty', 'rf_syncope', 'rf_seizure', 'rf_active_bleeding', 'rf_focal_neurologic_deficit', 'red_flags', 'red_flag_count', 'has_red_flag', 'target', 'audit_priority', 'split_role', 'camp_triage_label', 'camp_triage_score', 'camp_triage_rationale', 'critical_dx_override', 'camp_triage_label_final', 'label_changed_by_dx_override', 'camp_triage_rationale_final', 'audit_priority_final']


**Split & Features**

In [3]:
import re, gc
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import MaxAbsScaler, StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from scipy.sparse import hstack, csr_matrix

# ── Target ──
TARGET_MAP = {'LOW':0, 'MEDIUM':1, 'HIGH':2}
df['target'] = df['camp_triage_label_final'].map(TARGET_MAP)
df = df[df['target'].notna()].copy()
df['target'] = df['target'].astype(int)

# ── Stratified 80/10/10 split ──
np.random.seed(42)
splits = []
for cls in ['HIGH', 'MEDIUM', 'LOW']:
    idx = df[df['camp_triage_label_final'] == cls].index.tolist()
    np.random.shuffle(idx)
    n = len(idx); nt = int(n*0.10); nv = int(n*0.10)
    for i, ix in enumerate(idx):
        if i < nt:        splits.append((ix, 'TEST'))
        elif i < nt+nv:   splits.append((ix, 'VALIDATION'))
        else:             splits.append((ix, 'TRAIN'))
df['split'] = pd.Series({ix: sp for ix, sp in splits})

train_df = df[df['split']=='TRAIN'].reset_index(drop=True)
val_df   = df[df['split']=='VALIDATION'].reset_index(drop=True)
test_df  = df[df['split']=='TEST'].reset_index(drop=True)
y_train  = train_df['target'].values
y_val    = val_df['target'].values
y_test   = test_df['target'].values

print(f"Split:")
for name, s in [('TRAIN', train_df), ('VALIDATION', val_df), ('TEST', test_df)]:
    ct = s['camp_triage_label_final'].value_counts()
    print(f"  {name:<12} {len(s):>5,}  "
          f"HIGH={ct.get('HIGH',0)}  "
          f"MED={ct.get('MEDIUM',0)}  "
          f"LOW={ct.get('LOW',0)}")

# ── Class weights ──
cw = compute_class_weight('balanced', classes=np.array([0,1,2]), y=y_train)
class_weight_dict = {i: w for i, w in enumerate(cw)}
print(f"\nClass weights: { {k: round(v,3) for k,v in class_weight_dict.items()} }")

# ── Text cleaning ──
def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    text = re.sub(r'\|', ' ', text)
    text = re.sub(r'symptoms:\s*', '', text)
    text = re.sub(r'age:\s*(\d+)', r'age\1', text)
    text = re.sub(r'[^a-z0-9_ ]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

for s in [train_df, val_df, test_df]:
    s['text_clean'] = s['model_input_text'].apply(clean_text)

# ── Condition group rates ──
high_rate_map = (train_df.groupby('condition_group')['target']
                 .apply(lambda x: (x==2).mean()).to_dict())
med_rate_map  = (train_df.groupby('condition_group')['target']
                 .apply(lambda x: (x==1).mean()).to_dict())
low_rate_map  = (train_df.groupby('condition_group')['target']
                 .apply(lambda x: (x==0).mean()).to_dict())

# ── Vital imputation per class ──
VITAL_COLS = ['vital_hr','vital_spo2','vital_sbp',
              'vital_rr','vital_temp','pain_score']
for col in VITAL_COLS:
    for s in [train_df, val_df, test_df]:
        s[col] = pd.to_numeric(s[col], errors='coerce')
vital_medians = {}
for cls in [0,1,2]:
    sub = train_df[train_df['target']==cls]
    vital_medians[cls] = {c: sub[c].median() for c in VITAL_COLS}
for s in [train_df, val_df, test_df]:
    for cls in [0,1,2]:
        mask = s['target'] == cls
        for c in VITAL_COLS:
            s.loc[mask, c] = s.loc[mask, c].fillna(vital_medians[cls][c])

print(f"\nVital medians by class:")
for cls, name in [(0,'LOW'),(1,'MEDIUM'),(2,'HIGH')]:
    vals = '  '.join(f"{c.split('_')[1][:3]}={vital_medians[cls][c]:.1f}"
                     for c in VITAL_COLS)
    print(f"  {name:<8} {vals}")

# ── Clinical threshold flags ──
VT = {
    'vf_bradycardia':       ('vital_hr',   '<',  50),
    'vf_hypox_severe':      ('vital_spo2', '<',  90),
    'vf_hypox_moderate':    ('vital_spo2', '<',  94),
    'vf_hypotension':       ('vital_sbp',  '<',  90),
    'vf_hypertension_crit': ('vital_sbp',  '>',  180),
    'vf_tachypnea_severe':  ('vital_rr',   '>',  24),
    'vf_bradypnea':         ('vital_rr',   '<',  10),
    'vf_hypothermia':       ('vital_temp', '<',  35.0),
    'vf_high_fever':        ('vital_temp', '>',  38.5),
    'vf_pain_severe':       ('pain_score', '>=', 8),
    'vf_pain_none':         ('pain_score', '==', 0),
}

def apply_thresh(s):
    for flag, (col, op, val) in VT.items():
        v = pd.to_numeric(s[col], errors='coerce')
        if op == '>':    s[flag] = (v >  val).astype(float)
        elif op == '<':  s[flag] = (v <  val).astype(float)
        elif op == '>=': s[flag] = (v >= val).astype(float)
        elif op == '==': s[flag] = (v == val).astype(float)
        s[flag] = s[flag].fillna(0)
    age = s['age'].fillna(35)
    s['vf_elderly']        = (age >= 65).astype(float)
    s['vf_child']          = (age <   5).astype(float)
    s['vf_pain_low']       = (s['pain_score'] <= 3).astype(float).fillna(0)
    s['vf_critical_vital'] = s[['vf_hypox_severe','vf_hypotension',
                                 'vf_bradycardia','vf_hypothermia']].max(axis=1)
    s['vf_any_danger']     = s[['vf_hypox_severe','vf_hypotension',
                                 'vf_bradycardia','vf_hypothermia',
                                 'vf_tachypnea_severe','vf_bradypnea']].max(axis=1)
    s['vf_elder_x_pain']   = (age>=65).astype(float) * s['pain_score'].fillna(0)/10

for s in [train_df, val_df, test_df]:
    apply_thresh(s)

VF_COLS = list(VT.keys()) + ['vf_elderly','vf_child','vf_pain_low',
                              'vf_critical_vital','vf_any_danger','vf_elder_x_pain']

# ── TF-IDF ──
KEEP = {'of','in','and','or','not','no','with','without','upper','lower',
        'left','right','bilateral','acute','chronic'}
custom_stop = [w for w in ENGLISH_STOP_WORDS if w not in KEEP]
tfidf = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.90,
                        max_features=8000, sublinear_tf=True,
                        stop_words=custom_stop)
Xt  = tfidf.fit_transform(train_df['text_clean'])
Xvt = tfidf.transform(val_df['text_clean'])
Xtt = tfidf.transform(test_df['text_clean'])
print(f"\nTF-IDF vocabulary: {len(tfidf.vocabulary_):,} tokens")

# ── Condition group one-hot + prior rates ──
le_cg = LabelEncoder()
le_cg.fit(train_df['condition_group'])
def encode_cg(s):
    n = len(s)
    oh = np.zeros((n, len(le_cg.classes_)))
    for i, cg in enumerate(s['condition_group']):
        idx = np.where(le_cg.classes_ == cg)[0]
        if len(idx): oh[i, idx[0]] = 1
    hp = np.array([high_rate_map.get(cg, 0.26) for cg in s['condition_group']]).reshape(-1,1)
    mp = np.array([med_rate_map.get(cg,  0.33) for cg in s['condition_group']]).reshape(-1,1)
    lp = np.array([low_rate_map.get(cg,  0.41) for cg in s['condition_group']]).reshape(-1,1)
    return np.hstack([oh, hp, mp, lp])
cg_tr = encode_cg(train_df)
cg_v  = encode_cg(val_df)
cg_te = encode_cg(test_df)

# ── Structured features ──
KNOWN_FLAGS = ['breathing_difficulty','chest_pain','focal_neurologic_deficit',
               'active_bleeding','syncope','seizure']
RED_FLAG_COLS = ['has_red_flag','red_flag_count'] + [f'rf_{f}' for f in KNOWN_FLAGS]
for col in RED_FLAG_COLS:
    for s in [train_df, val_df, test_df]:
        if col not in s.columns: s[col] = 0
        s[col] = s[col].fillna(0)

def build_struct(s):
    rf    = s[RED_FLAG_COLS].values.astype(float)
    age   = s['age'].fillna(35)
    age_n = (age/100.0).values.reshape(-1,1)
    age_e = (age>=65).astype(float).values.reshape(-1,1)
    age_c = (age< 5).astype(float).values.reshape(-1,1)
    age_a = ((age>=5)&(age<65)).astype(float).values.reshape(-1,1)
    sym_c = s['text_clean'].str.split().str.len()
    hp = np.array([high_rate_map.get(cg,0.26) for cg in s['condition_group']]).reshape(-1,1)
    mp = np.array([med_rate_map.get(cg, 0.33) for cg in s['condition_group']]).reshape(-1,1)
    lp = np.array([low_rate_map.get(cg, 0.41) for cg in s['condition_group']]).reshape(-1,1)
    has_rf_inv  = (1 - s['has_red_flag'].values).reshape(-1,1)
    med_score   = (mp * age_a * has_rf_inv *
                   (sym_c.values/20.0).clip(0.1,0.8).reshape(-1,1))
    low_score   = (lp * has_rf_inv * (1-age_e)).reshape(-1,1)
    elder_x_hp  = (age_e.flatten() * hp.flatten()).reshape(-1,1)
    rf_x_hp     = (s['red_flag_count'].values * hp.flatten()).reshape(-1,1)
    vf          = s[VF_COLS].values.astype(float)
    pain        = s['pain_score'].fillna(5).values.reshape(-1,1) / 10.0
    pain_x_age  = (pain.flatten() * age_e.flatten()).reshape(-1,1)
    return np.hstack([rf, age_n, age_c, age_e, age_a,
                      (sym_c.values/20.0).reshape(-1,1),
                      hp, mp, lp, med_score, low_score,
                      elder_x_hp, rf_x_hp, vf, pain, pain_x_age])

st_tr = build_struct(train_df).astype('float32')
st_v  = build_struct(val_df).astype('float32')
st_te = build_struct(test_df).astype('float32')
print(f"Structured features: {st_tr.shape[1]} cols")

# ── Sparse full matrix (text + cg + struct) ──
X_tr = hstack([Xt,  csr_matrix(cg_tr), csr_matrix(st_tr)])
X_v  = hstack([Xvt, csr_matrix(cg_v),  csr_matrix(st_v)])
X_te = hstack([Xtt, csr_matrix(cg_te), csr_matrix(st_te)])
sc   = MaxAbsScaler()
X_tr_sc = sc.fit_transform(X_tr).astype('float32')
X_v_sc  = sc.transform(X_v).astype('float32')
X_te_sc = sc.transform(X_te).astype('float32')

# ── Dense matrix (cg + struct, for tree models) ──
Xd_tr = np.hstack([cg_tr, st_tr]).astype('float32')
Xd_v  = np.hstack([cg_v,  st_v]).astype('float32')
Xd_te = np.hstack([cg_te, st_te]).astype('float32')
sc_d  = StandardScaler()
Xd_tr_sc = sc_d.fit_transform(Xd_tr).astype('float32')
Xd_v_sc  = sc_d.transform(Xd_v).astype('float32')
Xd_te_sc = sc_d.transform(Xd_te).astype('float32')

del X_tr, X_v, X_te, Xt, Xvt, Xtt
del cg_tr, cg_v, cg_te, Xd_tr, Xd_v, Xd_te
gc.collect()

print(f"\nX_train sparse : {X_tr_sc.shape}")
print(f"X_train dense  : {Xd_tr_sc.shape}")

Split:
  TRAIN        6,064  HIGH=1586  MED=1978  LOW=2500
  VALIDATION     757  HIGH=198  MED=247  LOW=312
  TEST           757  HIGH=198  MED=247  LOW=312

Class weights: {0: np.float64(0.809), 1: np.float64(1.022), 2: np.float64(1.274)}

Vital medians by class:
  LOW      hr=84.0  spo=98.0  sbp=133.0  rr=18.0  tem=36.7  sco=1.0
  MEDIUM   hr=86.0  spo=98.0  sbp=137.5  rr=18.0  tem=36.7  sco=8.0
  HIGH     hr=101.0  spo=98.0  sbp=129.0  rr=20.0  tem=36.9  sco=2.0

TF-IDF vocabulary: 2,731 tokens
Structured features: 39 cols

X_train sparse : (6064, 2783)
X_train dense  : (6064, 52)


In [4]:
cols_to_drop = ['split_role', 'audit_priority', 'target',
                'camp_triage_label', 'camp_triage_score',
                'camp_triage_rationale', 'split']

df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
print(f"Remaining columns ({len(df.columns)}):")
print(list(df.columns))

Remaining columns (44):
['model_input_text', 'symptoms', 'age', 'severity', 'three_level_target', 'source_severity', 'condition_group', 'vital_hr', 'vital_spo2', 'vital_sbp', 'vital_rr', 'vital_temp', 'pain_score', 'has_vitals', 'injury_flag', 'data_source', 'immedr_raw', 'dx_cpr', 'dx_intubate', 'dx_bpap', 'dx_centline', 'dx_lactate', 'dx_cardenz', 'dx_abg', 'dx_cardmon', 'dx_ekg', 'dx_ivfluids', 'dx_cbc', 'dx_catscan', 'dx_resus', 'rf_chest_pain', 'rf_breathing_difficulty', 'rf_syncope', 'rf_seizure', 'rf_active_bleeding', 'rf_focal_neurologic_deficit', 'red_flags', 'red_flag_count', 'has_red_flag', 'critical_dx_override', 'camp_triage_label_final', 'label_changed_by_dx_override', 'camp_triage_rationale_final', 'audit_priority_final']


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (balanced_accuracy_score, recall_score, f1_score,
                             precision_score, classification_report,
                             confusion_matrix)
import lightgbm as lgb
import xgboost as xgb

LABEL_NAMES = ['LOW', 'MEDIUM', 'HIGH']

# ── OOF Stacking (5-fold) ──
print("OOF stacking (5-fold) ...")
N_SPLITS = 5
N_BASE   = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
n_tr = len(y_train)

oof    = np.zeros((n_tr,          N_BASE*3))
val_f  = np.zeros((len(y_val),   N_BASE*3))
test_f = np.zeros((len(y_test),  N_BASE*3))

for fold, (tr_idx, oof_idx) in enumerate(skf.split(np.zeros(n_tr), y_train)):
    print(f"  Fold {fold+1}/{N_SPLITS} ...", end=' ', flush=True)

    Xf  = X_tr_sc[tr_idx];  Xo  = X_tr_sc[oof_idx];  yf = y_train[tr_idx]
    Xdf = Xd_tr_sc[tr_idx]; Xdo = Xd_tr_sc[oof_idx]

    # SMOTE on dense matrix to balance toward MEDIUM size
    counts = dict(zip(*np.unique(yf, return_counts=True)))
    tgt    = int(counts.get(1, 1000) * 0.80)
    strat  = {}
    for cls in [0, 2]:
        if counts.get(cls, 0) < tgt:
            strat[cls] = max(tgt, counts.get(cls, 0) + 1)
    if strat:
        k = max(1, min(5, min(counts.values()) - 1))
        sm = SMOTE(sampling_strategy=strat, k_neighbors=k, random_state=fold)
        Xdf_sm, yf_sm = sm.fit_resample(Xdf, yf)
    else:
        Xdf_sm, yf_sm = Xdf, yf

    sw = compute_sample_weight(class_weight_dict, yf_sm)

    # Base models
    mf = []

    # LR × 3 (sparse)
    for name, C in [('lr_a', 0.3), ('lr_b', 0.1), ('lr_c', 0.05)]:
        m = LogisticRegression(multi_class='multinomial', solver='saga',
                               max_iter=300, C=C,
                               class_weight=class_weight_dict,
                               random_state=42, n_jobs=-1)
        m.fit(Xf, yf)
        mf.append((name, m, 'sparse'))

    # LightGBM (dense)
    lgbm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1,
                               max_depth=6, num_leaves=31,
                               min_child_samples=5,
                               subsample=0.8, colsample_bytree=0.8,
                               class_weight=class_weight_dict,
                               random_state=fold, verbosity=-1, n_jobs=-1)
    lgbm.fit(Xdf_sm, yf_sm, sample_weight=sw)
    mf.append(('lgb', lgbm, 'dense'))

    # XGBoost (dense)
    xgbm = xgb.XGBClassifier(n_estimators=300, max_depth=5,
                              learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8,
                              min_child_weight=3, gamma=0.1,
                              num_class=3, objective='multi:softprob',
                              eval_metric='mlogloss',
                              random_state=fold, verbosity=0,
                              n_jobs=-1, tree_method='hist')
    xgbm.fit(Xdf_sm, yf_sm, sample_weight=sw)
    mf.append(('xgb', xgbm, 'dense'))

    # Collect OOF and held-out probabilities
    for i, (nm, m, ft) in enumerate(mf):
        Xp  = Xo       if ft == 'sparse' else Xdo
        Xpv = X_v_sc   if ft == 'sparse' else Xd_v_sc
        Xpt = X_te_sc  if ft == 'sparse' else Xd_te_sc
        oof[oof_idx,  i*3:(i+1)*3] =  m.predict_proba(Xp)
        val_f[:,      i*3:(i+1)*3] += m.predict_proba(Xpv)
        test_f[:,     i*3:(i+1)*3] += m.predict_proba(Xpt)

    print("done")

val_avg  = val_f  / N_SPLITS
test_avg = test_f / N_SPLITS

# ── Meta-learner ──
print("\nMeta-learner sweep ...")
mXtr = np.hstack([oof,      st_tr])
mXv  = np.hstack([val_avg,  st_v])
mXte = np.hstack([test_avg, st_te])

best = {'bal': 0}
for C in [0.05, 0.1, 0.2, 0.5, 1.0]:
    for cw2 in [
        {0:1.0,1:1.0,2:2.0}, {0:1.5,1:1.0,2:2.5},
        {0:1.0,1:1.5,2:2.0}, {0:1.5,1:1.5,2:1.5},
        {0:1.0,1:1.0,2:3.0}, {0:2.0,1:1.5,2:1.5},
        {0:1.0,1:2.0,2:2.0}, {0:1.5,1:2.0,2:1.5},
    ]:
        m = LogisticRegression(C=C, max_iter=500,
                               class_weight=cw2, random_state=42)
        m.fit(mXtr, y_train)
        yp  = m.predict(mXv)
        bal = balanced_accuracy_score(y_val, yp)
        if bal > best['bal']:
            recs = recall_score(y_val, yp, average=None)
            best = {'bal': bal, 'C': C, 'cw': cw2, 'model': m,
                    'hrc': recs[2], 'mrc': recs[1], 'lrc': recs[0]}

meta = best['model']
print(f"  Best  C={best['C']}  val_bal={best['bal']:.4f}  "
      f"HIGH={best['hrc']:.3f}  MED={best['mrc']:.3f}  LOW={best['lrc']:.3f}")

# ── Threshold sweep ──
print("\nThreshold sweep ...")
pv = meta.predict_proba(mXv)
pt = meta.predict_proba(mXte)

best_t = {'bal': 0, 't_high': 0.35, 't_low': 0.40, 'hrc': 0, 'mrc': 0}
for t_high in np.arange(0.10, 0.65, 0.01):
    for t_low in np.arange(0.15, 0.65, 0.01):
        if t_high <= t_low: continue
        ph = pv[:,2]; pl = pv[:,0]
        yp = np.argmax(pv, axis=1).copy()
        yp[ph >= t_high] = 2
        yp[(pl >= t_low) & (ph < t_high)] = 0
        yp[val_df['has_red_flag'].values == 1] = np.maximum(
            yp[val_df['has_red_flag'].values == 1], 2)
        hrc = recall_score(y_val, yp, average=None)[2]
        if hrc < 0.80: continue
        bal = balanced_accuracy_score(y_val, yp)
        if bal > best_t['bal']:
            mrc = recall_score(y_val, yp, average=None)[1]
            best_t = dict(bal=bal, t_high=t_high, t_low=t_low,
                          hrc=hrc, mrc=mrc)

TH = best_t['t_high']
TL = best_t['t_low']
print(f"  t_high={TH:.3f}  t_low={TL:.3f}  "
      f"BalAcc={best_t['bal']:.4f}  "
      f"HIGHrec={best_t['hrc']:.4f}  MEDrec={best_t['mrc']:.4f}")

# ── Prediction function with all override layers ──
HIGH_COMPLAINT_PATTERNS = [
    'crush injury', 'head trauma', 'heart failure',
    'pregnancy complication', 'traumatic brain injury',
    'head injury', 'drowning', 'electric shock',
    'gunshot wound', 'stab wound', 'chest trauma',
    'spinal injury', 'paralysis weakness one side',
    'stroke', 'seizure convulsion', 'anaphylaxis',
]
COMPLAINT_PATTERN = '|'.join(HIGH_COMPLAINT_PATTERNS)

def predict_final(mp, df_s, th, tl):
    ph = mp[:,2]; pl = mp[:,0]
    yp = np.argmax(mp, axis=1).copy()
    # Threshold cascade
    yp[ph >= th] = 2
    yp[(pl >= tl) & (ph < th)] = 0
    # Layer 1 — red flag text override
    yp[df_s['has_red_flag'].values == 1] = np.maximum(
        yp[df_s['has_red_flag'].values == 1], 2)
    # Layer 2 — dx resuscitation override
    for col in ['dx_resus', 'dx_intubate', 'dx_centline']:
        if col in df_s.columns:
            yp[df_s[col].fillna(0).values == 1] = 2
    # Layer 3 — complaint-based override
    yp[df_s['symptoms'].str.contains(
        COMPLAINT_PATTERN, case=False, na=False).values] = 2
    return yp

# ── Evaluation ──
def eval_set(name, mp, df_s, y_true):
    yp = predict_final(mp, df_s, TH, TL)
    cm = confusion_matrix(y_true, yp)
    bal  = balanced_accuracy_score(y_true, yp)
    mf1  = f1_score(y_true, yp, average='macro')
    recs = recall_score(y_true, yp, average=None)
    pres = precision_score(y_true, yp, average=None, zero_division=0)
    f1s  = f1_score(y_true, yp, average=None)
    print(f"\n{'═'*58}")
    print(f"  {name}")
    print(f"{'═'*58}")
    print(classification_report(y_true, yp,
                                target_names=LABEL_NAMES, digits=3))
    print(f"  Confusion matrix:")
    print(f"  {'':10} {'LOW':>6} {'MED':>8} {'HIGH':>8}")
    for i, l in enumerate(LABEL_NAMES):
        print(f"  {l:<10} {cm[i,0]:>6} {cm[i,1]:>8} {cm[i,2]:>8}")
    print(f"\n  Balanced accuracy : {bal:.4f}")
    print(f"  Macro F1          : {mf1:.4f}")
    for i, nc in enumerate(LABEL_NAMES):
        print(f"  {nc:<8}  rec={recs[i]:.3f}  "
              f"prec={pres[i]:.3f}  f1={f1s[i]:.3f}")
    print(f"\n  HIGH→LOW errors   : {cm[2,0]}  ← must be 0")
    return bal, recs[2], recs[1], yp

b_v, h_v, m_v, _ = eval_set("VALIDATION", pv, val_df,  y_val)
b_t, h_t, m_t, _ = eval_set("TEST SET",   pt, test_df, y_test)

print(f"\n{'═'*58}")
print(f"  FINAL RESULTS")
print(f"{'═'*58}")
print(f"  Dataset      : nhamcs_2022_camp_relabeled_v2.csv")
print(f"  Labels       : camp_triage_label_final")
print(f"  Records      : 7,578  (TRAIN 6,064 / VAL 757 / TEST 757)")
print(f"  Balanced acc : {b_t:.4f}")
print(f"  Macro F1     : {f1_score(_, _, average='macro'):.4f}")
print(f"  HIGH recall  : {h_t:.4f}")
print(f"  MED recall   : {m_t:.4f}")
print(f"  TH={TH:.3f}  TL={TL:.3f}")

OOF stacking (5-fold) ...
  Fold 1/5 ... done
  Fold 2/5 ... done
  Fold 3/5 ... done
  Fold 4/5 ... done
  Fold 5/5 ... done

Meta-learner sweep ...
  Best  C=0.2  val_bal=0.9121  HIGH=0.869  MED=0.903  LOW=0.965

Threshold sweep ...
  t_high=0.470  t_low=0.460  BalAcc=0.8926  HIGHrec=0.8838  MEDrec=0.8421

══════════════════════════════════════════════════════════
  VALIDATION
══════════════════════════════════════════════════════════
              precision    recall  f1-score   support

         LOW      0.949     0.946     0.947       312
      MEDIUM      0.907     0.826     0.864       247
        HIGH      0.828     0.924     0.874       198

    accuracy                          0.901       757
   macro avg      0.894     0.899     0.895       757
weighted avg      0.903     0.901     0.901       757

  Confusion matrix:
                LOW      MED     HIGH
  LOW           295        6       11
  MEDIUM         16      204       27
  HIGH            0       15      183

  Bal

In [7]:
import joblib, json, os

EXPORT = f'{PROJ}/model_artefacts'
os.makedirs(EXPORT, exist_ok=True)

# Save artefacts
joblib.dump(tfidf,  f'{EXPORT}/tfidf_vectorizer.pkl')
joblib.dump(sc,     f'{EXPORT}/scaler_maxabs.pkl')
joblib.dump(sc_d,   f'{EXPORT}/scaler_standard.pkl')
joblib.dump(le_cg,  f'{EXPORT}/label_encoder_cg.pkl')
joblib.dump(meta,   f'{EXPORT}/meta_learner.pkl')

# Save config
config = {
    'model_version':        'v2_camp_relabelled',
    'dataset':              'nhamcs_2022_camp_relabeled_v2.csv',
    'label_column':         'camp_triage_label_final',
    'classes':              ['LOW', 'MEDIUM', 'HIGH'],
    'label_map':            {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2},
    'n_base_models':        5,
    'n_folds':              5,
    'threshold_high':       round(float(TH), 3),
    'threshold_low':        round(float(TL), 3),
    'test_balanced_acc':    0.8944,
    'test_macro_f1':        0.8887,
    'test_high_recall':     0.9596,
    'test_med_recall':      0.7814,
    'test_low_recall':      0.9423,
    'high_to_low_errors':   0,
    'override_layers': {
        'layer_1': 'red_flag text match → floor to HIGH',
        'layer_2': 'dx_resus / dx_intubate / dx_centline → force HIGH',
        'layer_3': 'complaint pattern match → force HIGH',
    },
    'high_complaint_patterns': HIGH_COMPLAINT_PATTERNS,
    'dx_override_cols':     ['dx_resus', 'dx_intubate', 'dx_centline'],
    'vital_medians':        {str(k): {c: round(v, 1) for c, v in d.items()}
                             for k, d in vital_medians.items()},
    'condition_groups':     list(le_cg.classes_),
    'high_rate_map':        {k: round(v, 4) for k, v in high_rate_map.items()},
    'med_rate_map':         {k: round(v, 4) for k, v in med_rate_map.items()},
    'low_rate_map':         {k: round(v, 4) for k, v in low_rate_map.items()},
    'vf_cols':              VF_COLS,
    'rf_cols':              RED_FLAG_COLS,
    'vital_cols':           VITAL_COLS,
}
with open(f'{EXPORT}/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Verify
print(f"{'='*50}")
print(f"  MODEL ARTEFACTS EXPORTED")
print(f"{'='*50}")
for fname in sorted(os.listdir(EXPORT)):
    size = os.path.getsize(f'{EXPORT}/{fname}') / 1024
    print(f"  {fname:<38} {size:>7.1f} KB")

print(f"\n  Path : {EXPORT}")
print(f"  TH   : {TH:.3f}")
print(f"  TL   : {TL:.3f}")
print(f"\n Export complete")

  MODEL ARTEFACTS EXPORTED
  label_encoder_cg.pkl                       0.6 KB
  meta_learner.pkl                           2.2 KB
  model_config.json                          3.6 KB
  scaler_maxabs.pkl                         43.9 KB
  scaler_standard.pkl                        1.8 KB
  tfidf_vectorizer.pkl                     133.2 KB

  Path : /content/drive/MyDrive/severity_queue/model_artefacts
  TH   : 0.470
  TL   : 0.460

 Export complete


In [8]:
from google.colab import files
import os

EXPORT = f'{PROJ}/model_artefacts'

for fname in sorted(os.listdir(EXPORT)):
    fpath = f'{EXPORT}/{fname}'
    print(f"Downloading {fname} ...")
    files.download(fpath)

print("\n All artefacts downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 All artefacts downloaded
